# View Purge Behavior Test — purge-view-metadata-on-drop

**Goal:** Document what happens when a view is dropped under each config value,
with confirmed-working MinIO credentials.

This answers the manager's question precisely:
> What happens when you purge a view inside a catalog where
> `polaris.config.purge-view-metadata-on-drop = true` (the default)?

## Test matrix
```
Case 1: purge-view-metadata-on-drop = TRUE  (default)
        → does the view metadata file get deleted from MinIO?
        → does the drop succeed or 403?

Case 2: purge-view-metadata-on-drop = FALSE
        → drop is metadata-only (PostgreSQL), MinIO untouched?

We observe MinIO directly to confirm whether the .metadata.json
view file is actually deleted.
```


In [ ]:
import sys
sys.path.insert(0, '.')
from polaris_test_utils import *
import s3fs, time

tok = root_token()
print("✅ root token")

TEST_CATALOG  = "view-purge-test"
TEST_NS       = "vp-ns"
TEST_LOCATION = "s3a://data-catalog-bucket/view-purge-test/"

MINIO_ENDPOINT   = "http://192.168.139.2:9000"
MINIO_ACCESS_KEY = "admin"
MINIO_SECRET_KEY = "bhEenDzTEAVPKbukhuY0tRY4WjpZXzh0"

fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY, secret=MINIO_SECRET_KEY,
    use_ssl=False, client_kwargs={"endpoint_url": MINIO_ENDPOINT}
)

def list_view_files(view_name):
    """List MinIO files under a view's location."""
    prefix = f"data-catalog-bucket/view-purge-test/{TEST_NS}/{view_name}/"
    try:
        files = fs.ls(prefix, detail=False)
        return files
    except FileNotFoundError:
        return []


In [ ]:
# Cleanup any leftover from prior runs
print("Cleanup...")
catalogs = requests.get(f"{BASE_MGMT}/catalogs", headers=h(tok)).json().get("catalogs", [])
if any(c["name"] == TEST_CATALOG for c in catalogs):
    # enable purge + grant so we can fully clear
    cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
    merged = {**cat.get("properties", {}),
              "polaris.config.drop-with-purge.enabled": "true",
              "polaris.config.purge-view-metadata-on-drop": "false"}
    requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
    requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}/catalog-roles/catalog_admin/grants",
                 headers=h(tok), json={"grant": {"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})
    for ns in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces", headers=h(tok)).json().get("namespaces", []):
        nsn = ns[0] if isinstance(ns, list) else ns
        for v in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
            requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
        requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}", headers=h(tok))
    requests.delete(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}?purgeRequested=true", headers=h(tok))
    print("  removed old test catalog")
print("✅ clean")


In [ ]:
# Create catalog with purge-view-metadata-on-drop = TRUE (the DEFAULT we're testing)
print("Create catalog (purge-view-metadata-on-drop = TRUE)")
r = requests.post(f"{BASE_MGMT}/catalogs", headers=h(tok), json={
    "catalog": {
        "name": TEST_CATALOG,
        "type": "INTERNAL",
        "properties": {
            "default-base-location": TEST_LOCATION,
            "polaris.config.purge-view-metadata-on-drop": "true",  # explicit, = default
        },
        "storageConfigInfo": {
            "storageType": "S3",
            "allowedLocations": [TEST_LOCATION],
            "pathStyleAccess": True,
            "endpoint": "http://192.168.139.2:9000",
            "endpointInternal": "http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000",
        },
    }
})
print_response(r, "Create catalog")

# Grant content management so we have view privileges
requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}/catalog-roles/catalog_admin/grants",
             headers=h(tok), json={"grant": {"type":"catalog","privilege":"CATALOG_MANAGE_CONTENT"}})

# Namespace
requests.post(f"{BASE_CAT}/{TEST_CATALOG}/namespaces", headers=h(tok),
              json={"namespace": [TEST_NS], "properties": {}})
print("✅ catalog + namespace ready")


## Case 1 — Drop view with purge-view-metadata-on-drop = TRUE

In [ ]:
# Create a view, confirm its metadata file exists in MinIO, then drop it
VIEW1 = "view-true"
print(f"Create view: {VIEW1}")
r = requests.post(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views",
    headers=h(tok),
    json={
        "name": VIEW1,
        "default-namespace": [TEST_NS],
        "schema": {"type": "struct", "schema-id": 0,
                   "fields": [{"id": 1, "name": "id", "type": "long", "required": True}]},
        "view-version": {
            "version-id": 1, "timestamp-ms": int(time.time()*1000), "schema-id": 0,
            "default-namespace": [TEST_NS], "summary": {"engine": "spark"},
            "representations": [{"type": "sql", "sql": "SELECT 1 AS id", "dialect": "spark"}],
        },
        "location": f"{TEST_LOCATION}{TEST_NS}/{VIEW1}/",
    }
)
print_response(r, "Create view")


In [ ]:
# Confirm the view's metadata file exists in MinIO BEFORE drop
print("MinIO files for view BEFORE drop:")
files_before = list_view_files(VIEW1)
for f in files_before:
    print(f"  {f}")
print(f"  total: {len(files_before)} file(s)")


In [ ]:
# Drop the view — purge-view-metadata-on-drop is TRUE
# Question: does this succeed? does it delete the MinIO metadata file?
print(f"Drop view {VIEW1} (no purgeRequested flag)")
r = requests.delete(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views/{VIEW1}",
    headers=h(tok)
)
print_response(r, "Drop view (config=true)")
print(f"\nDrop status: {r.status_code}")


In [ ]:
# Confirm MinIO state AFTER drop
import time as _t
_t.sleep(2)  # allow async cleanup if any
print("MinIO files for view AFTER drop:")
files_after = list_view_files(VIEW1)
for f in files_after:
    print(f"  {f}")
print(f"  total: {len(files_after)} file(s)")
print()
if len(files_before) > 0 and len(files_after) == 0:
    print("✅ RESULT: metadata file WAS deleted from MinIO (purge-view-metadata=true worked)")
elif len(files_before) > 0 and len(files_after) > 0:
    print("ℹ️  RESULT: metadata file REMAINS in MinIO despite config=true")
elif len(files_before) == 0:
    print("ℹ️  No metadata file was created in MinIO to begin with")


## Case 2 — Drop view with purge-view-metadata-on-drop = FALSE

In [ ]:
# Switch catalog config to FALSE
print("Set purge-view-metadata-on-drop = FALSE")
cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
merged = {**cat.get("properties", {}), "polaris.config.purge-view-metadata-on-drop": "false"}
r = requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
                 json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
print(f"  update: {r.status_code}")
print(f"  config now: {r.json().get('properties', {}).get('polaris.config.purge-view-metadata-on-drop')}")


In [ ]:
# Create a second view, confirm metadata exists, drop it
VIEW2 = "view-false"
print(f"Create view: {VIEW2}")
requests.post(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views",
    headers=h(tok),
    json={
        "name": VIEW2,
        "default-namespace": [TEST_NS],
        "schema": {"type": "struct", "schema-id": 0,
                   "fields": [{"id": 1, "name": "id", "type": "long", "required": True}]},
        "view-version": {
            "version-id": 1, "timestamp-ms": int(time.time()*1000), "schema-id": 0,
            "default-namespace": [TEST_NS], "summary": {"engine": "spark"},
            "representations": [{"type": "sql", "sql": "SELECT 1 AS id", "dialect": "spark"}],
        },
        "location": f"{TEST_LOCATION}{TEST_NS}/{VIEW2}/",
    }
)

files_before2 = list_view_files(VIEW2)
print(f"MinIO files BEFORE drop: {len(files_before2)}")
for f in files_before2: print(f"  {f}")


In [ ]:
# Drop view with config=false
print(f"Drop view {VIEW2}")
r = requests.delete(
    f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{TEST_NS}/views/{VIEW2}",
    headers=h(tok)
)
print_response(r, "Drop view (config=false)")

import time as _t
_t.sleep(2)
files_after2 = list_view_files(VIEW2)
print(f"\nMinIO files AFTER drop: {len(files_after2)}")
for f in files_after2: print(f"  {f}")
print()
if len(files_before2) > 0 and len(files_after2) > 0:
    print("✅ RESULT: metadata file REMAINS (config=false → metadata-only drop)")
elif len(files_before2) > 0 and len(files_after2) == 0:
    print("ℹ️  RESULT: metadata file deleted despite config=false")
else:
    print("ℹ️  No metadata file created to begin with")


## Summary of findings

In [ ]:
# Print comparison table
print("="*60)
print("VIEW PURGE BEHAVIOR — TEST RESULTS")
print("="*60)
print()
print(f"Case 1: purge-view-metadata-on-drop = TRUE")
print(f"  drop status:        {r.status_code if False else 'see Case 1 cell'}")
print(f"  MinIO before/after: {len(files_before)} → {len(files_after)}")
print()
print(f"Case 2: purge-view-metadata-on-drop = FALSE")
print(f"  MinIO before/after: {len(files_before2)} → {len(files_after2)}")
print()
print("Interpretation:")
print("  TRUE  → Polaris deletes the view metadata file from MinIO on drop")
print("  FALSE → Polaris leaves the metadata file; only PostgreSQL record removed")
print()
print("Note: views hold only a small metadata file (the query definition),")
print("      never Parquet data files. So this only affects that one small file.")


In [ ]:
# Final cleanup
print("Final cleanup...")
cat = requests.get(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok)).json()
merged = {**cat.get("properties", {}),
          "polaris.config.drop-with-purge.enabled": "true",
          "polaris.config.purge-view-metadata-on-drop": "false"}
requests.put(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}", headers=h(tok),
             json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
for ns in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces", headers=h(tok)).json().get("namespaces", []):
    nsn = ns[0] if isinstance(ns, list) else ns
    for v in requests.get(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
        requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}/views/{v['name']}?purgeRequested=true", headers=h(tok))
    requests.delete(f"{BASE_CAT}/{TEST_CATALOG}/namespaces/{nsn}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/catalogs/{TEST_CATALOG}?purgeRequested=true", headers=h(tok))
print("✅ test catalog removed")
